# 推理验证

SFT 训练完成，现在来验证模型是否真正学会了 Wordle 的格式规范。本节将启动推理服务器，用 vf-eval 进行标准化评测，对比 SFT 前后的效果。

---

## 推理服务器

`scripts/infer_server.py` 是一个极简的 OpenAI 兼容推理服务器，用 `transformers` + `torch_npu` 直接加载 SFT 权重，监听 8000 端口。

### 工作流程

![Inference Server Flow](./images/inference-server-flow.png)


### 启动服务器


In [46]:
import os
original_dir = os.getcwd()
%cd /mnt/workspace/torchtitan-npu
%env BASH_ENV=/home/developer/Ascend/cann/set_env.sh

In [59]:
%%bash
# (可选）终止已存在的 infer_server.py 进程，避免端口 8000 被占用
pkill -f infer_server.py || true


In [61]:
%%bash
# 启动推理服务器（后台运行，日志重定向避免阻塞 cell）
pip install transformers
pip install torchvision==0.27.0+cpu --index-url https://download.pytorch.org/whl/cpu
python3 scripts/infer_server.py \
  --model ./outputs/checkpoint_wordle_sft/step-20 \
  --port 8000 > /tmp/infer_server.log 2>&1 &

# 等待服务器启动
sleep 10

# 验证服务器是否正常
curl http://localhost:8000/health
# → {"status": "ok"}



---

## vf-eval 评测

`vf-eval` 是 Prime-RL 使用的 `verifiers` 包提供的命令行评测工具。它加载 Wordle 游戏环境，向推理服务器发送多轮对话请求，模拟完整游戏过程并计算 reward。

### 安装评测环境

在 CANNLab 环境内：

In [ ]:
%%bash
# 安装 Wordle 游戏环境 + vf-eval（NLTK 语料库已由 02.03 准备）
set -euo pipefail

PRIMERL_DIR=./prime-rl
if [ -d "$PRIMERL_DIR/.git" ]; then
  echo "Using existing prime-rl at: $PRIMERL_DIR"
else
  git clone https://gitcode.com/gh_mirrors/pr/prime-rl.git "$PRIMERL_DIR"
fi
cd "$PRIMERL_DIR"
git checkout 188192ce64b2b7acf82e83ae36cfb0632bebde5b

VERIFIERS_REV=d822f6aca7a967fc6698b1d595524c6278d84a5c

if [ ! -e deps/verifiers/.git ] ||
    [ "$(git -C deps/verifiers rev-parse HEAD 2>/dev/null || true)" != "$VERIFIERS_REV" ]; then
  rm -rf deps/verifiers
  git init deps/verifiers
  git -C deps/verifiers remote add origin \
    https://gitcode.com/GitHub_Trending/ver/verifiers.git
  git -C deps/verifiers fetch --depth=1 origin "$VERIFIERS_REV"
  git -C deps/verifiers checkout --detach FETCH_HEAD
fi

# Prime-RL Wordle verifier 需要自己的 venv 环境。
uv venv .venv-wordle-legacy --clear
source .venv-wordle-legacy/bin/activate

uv pip install -e deps/verifiers --index-url https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple
uv pip install -e deps/verifiers/environments/wordle --index-url https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple

echo '=== 验证 ==='
echo "vf-eval: $(which vf-eval)"
echo "prime-rl revision: $(git rev-parse HEAD)"
echo "verifiers revision: $(git -C deps/verifiers rev-parse HEAD)"



### 评测工作流程

![vf-eval Game Flow](./images/vf-eval-game-flow.png)


### 运行评测


In [56]:
%%bash

set -euo pipefail

PRIMERL_DIR=./prime-rl
source "$PRIMERL_DIR/.venv-wordle-legacy/bin/activate"
vf-eval wordle \
  --num-examples 4 \
  --rollouts-per-example 2 \
  --api-base-url http://127.0.0.1:8000/v1 \
  --max-concurrent 1 \
  --verbose \
  --temperature 0.6 \
  --save-results \
  



| 参数 | 说明 |
|------|------|
| `--provider openai` | 使用 OpenAI 兼容 API |
| `--api-base-url` | 推理服务器地址 |
| `--num-examples 4` | 选择 4 个评测任务 |
| `--rollouts-per-example 2` | 每个任务采样 2 条 rollout，共 60 条 |
| `--max-concurrent 1` | 串行请求（避免单线程服务器过载） |
| `--max-tokens 1024` | 每轮最多生成 1024 tokens |
| `--temperature 0.6` | 采样温度（>0 产生多样性） |
| `--save-results` | 保存逐条结果和汇总，作为结论证据 |

---

### 结果分析
基线模型与 SFT 后的参考性能对比：

| 指标 | Qwen3-1.7B 参考值 | Wordle SFT 参考值 | 变化 |
|------|-------------------|-------------------|------|
| 平均 `format_reward` | 0.6 | 1.0 | +0.4 |
| `correct_answer` 均值 | 0.0 | 0.0 | 持平 |
| 平均 `partial_answer` | 0.0 | 0.25 | +0.25 |
| 平均 `length_bonus` | 0.0 | 0.0 | 持平 |
| 平均 `reward` | 0.22 | 0.4 | +0.18 |

1. **格式正确性显著提升**：`format_reward` 从 0.6 提升至 1.0，说明 SFT 阶段成功让模型学会了符合 Wordle 环境的 XML 交互格式，这与“SFT学习格式”的目标一致。

2. **部分正确性有所改善**：`partial_answer` 从 0 提高到 0.25，表明模型开始能够生成包含正确字母（G/Y 反馈）的猜测，逐步掌握游戏逻辑，但猜中率仍然为 0（`correct_answer` 未变）。

3. **整体奖励上升**：平均总奖励从 0.22 增至 0.4，增益主要来自格式和部分正确性的改进，但最终获胜率（length_bonus 仍为 0）未见提升，说明模型尚未能完整解出单词。

4. **与官方基线对比**： Qwen3-1.7B 基线（0.22）接近官方报告的 ~0.2，SFT 后（0.4）也基本符合预期。预计在 RL 阶段后平均奖励可达 ~1.5 且胜率约 60%，表明当前 SFT 模型是 RL 的基座，需要 RL 提升其策略能力。


#### 结论

- **SFT 达到预期目标**：经过监督微调，模型已充分掌握了课程要求的回复格式，能够稳定生成环境可解析的 `<guess>` 字段，平均格式分达到理想水平；同时，模型在训练过程中也获得了一定的上下文推理能力，为后续交互奠定了基础。
- **策略提升仍需 RL**：尽管 SFT 在格式层面表现合格，但其猜词策略尚未稳定（6 轮猜中率仍偏低），表明仅靠模仿学习难以充分优化决策。因此，下一步将执行多轮强化学习（RL），利用 Wordle 环境的逐轮 G/Y/X 反馈进行策略优化，预期可显著提升 `correct_answer` 和 `length_bonus`，进而提高总奖励与整体胜率。


下一章，我们将通过融合算子优化训练性能，在不改变模型行为的前提下提升训练效率。

## 练习

1. （判断题）仅凭平均 `format_reward` 上升或 `num_turns` 变长，就能证明模型已经理解 G/Y/X 规则。

2. （单选题）如果 SFT 的平均 `format_reward` 上升，但 `correct_answer` 没有上升，最稳妥的结论是什么？
    A. 模型一定没有学到任何内容
    B. 当前评测显示格式分改善，但没有显示猜中率改善；不能仅据此断言策略原因
    C. 评测脚本一定有 bug
    D. 只要继续训练，必然由 RL 解决

3. （单选题）当前 Wordle Rubric 中，哪个组件的 Rubric 权重是 0.2？
    A. correct_answer
    B. partial_answer
    C. length_bonus
    D. format_reward


In [51]:
%cd $original_dir
!cat ./answer/03.04_answer.txt